# Cambridge, MA — Land Value Tax (Split-Rate) Model

**Jurisdiction:** City of Cambridge, Middlesex County, Massachusetts (FIPS 25017)
**Fiscal year modeled:** FY2026 (assessments as of 1/1/2025; rates set 10/2025)
**Reform:** Revenue-neutral **4:1 split-rate** (land taxed 4x improvements), **city levy only**, **all existing exemptions preserved**.

Massachusetts is a **classified-tax** state: Cambridge sets separate class tax rates each year.
FY2026 rates (City of Cambridge FY26 Tax Rate Letter, Oct 6 2025):

| Class | Rate / $1,000 |
|---|---|
| Residential | **$6.67** |
| Commercial / Industrial / Personal (CIP) | **$14.07** |

**Official FY26 property tax levy: $678,852,471** (FY26 Tax Rate Letter, Table I, "Actual Property Tax Levy").
Levy split: residential 33.801% / CIP 66.199%. Total taxable assessed value $73,111,556,293
(Residential $41,167,007,215; Commercial $12,661,975,602; Industrial $16,726,820,278; Personal Property $2,555,753,198).

Because a land value tax falls on **real property** (land + buildings), personal property
($2.556B, taxed at the CIP rate ~= $35.96M) is outside the LVT base. The parcel-level
revenue-neutrality target is therefore the **real-property levy** =
$678,852,471 - $35,959,447 = **~$642,893,024**.

To preserve the existing structure (Q3 = keep exemptions/structure, change only the rate split),
the split-rate is modeled **within each class** (residential and CIP) independently — a classified
dual split-rate (the Rochester dual-rate precedent) — so the 33.8% / 66.2% class shares are held
fixed and each class shifts internally toward land.

### Data
- **Assessing values:** CambridgeGIS *Property Database* (FeatureServer, FY2026, `YearOfAssessment=2026`)
  — land value, building value, assessed value, state class code, per-parcel residential-exemption flag.
- **Geometry:** CambridgeGIS *FY2026 Parcels* polygons, joined on `GISID` -> `ML`.

### Key data-cleaning decisions (see Sections 3 & 5)
1. **De-duplicate by `PID`.** Multi-building parcels repeat the *parcel-total* value on each
   `BldgNum` sub-record; keeping one row per `PID` avoids multiplying value by building count.
2. **Classification by leading digit, not a naive threshold.** Cambridge appends 4-digit
   residential variants (1014, 1021 luxury condo, 1029, 1091, 1094, 1095, 1322, ...) that are
   numerically >900 but are residential; true exempts are 900-999 or 9000-9999.
3. **Condo land imputation.** Condo units carry `$0` land / full value in "building" (token-land
   artifact) with `LandArea=0`; the land sits on the class-199 CONDO-BLDG master (lot area, $0 value).
   Land is imputed density-aware: building land = lot area x citywide residential land $/sqft,
   distributed to units by value share (rewards dense buildings — the correct LVT signal).


## Section 1 — Imports and Setup

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'cambridge'
STATE_FIPS = '25'
COUNTY_FIPS = '017'          # Middlesex County
MODEL_TYPE = 'split_rate:4.0'   # classified dual split-rate (residential + CIP)
LAND_IMPROVEMENT_RATIO = 4.0

# FY2026 official policy figures (City of Cambridge FY26 Tax Rate Letter, 2025-10-06)
RES_RATE = 6.67 / 1000.0
CIP_RATE = 14.07 / 1000.0
RES_EXEMPTION = 510208          # FY26 owner-occupant residential exemption ($ off assessed value)
OFFICIAL_TOTAL_LEVY = 678_852_471
PERSONAL_PROP_AV = 2_555_753_198
PERSONAL_PROP_TAX = PERSONAL_PROP_AV * CIP_RATE
REAL_PROPERTY_TARGET = OFFICIAL_TOTAL_LEVY - PERSONAL_PROP_TAX   # ~= $642.9M

# Parcel-map identity columns
PARCEL_ID_COL = 'MapLot'
OWNER_NAME_COL = 'Owner_Name'
OWNER_ADDRESS_COL = 'Address'
PARCEL_URL_TEMPLATE = None       # Cambridge property viewer is session-based (no stable deep link)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

ORG = 'https://services1.arcgis.com/WnzC35krSYGuYov4/arcgis/rest/services'


## Section 2 — Fetch / Load Parcel Data

Two sources, cached in `data/`: the FY2026 Property Database (assessing table, no geometry)
and the FY2026 Parcels polygons (geometry).

In [ ]:
PROP_PATH = DATA_DIR / 'property_database.parquet'
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

def _fetch_table(url, fields='*'):
    rows, off = [], 0
    while True:
        p = dict(where='1=1', outFields=fields, returnGeometry='false',
                 resultOffset=off, resultRecordCount=2000, f='json')
        d = requests.get(url, params=p, timeout=120).json()
        feats = d.get('features', [])
        if not feats:
            break
        rows.extend(f['attributes'] for f in feats)
        off += len(feats)
        if len(feats) < 2000 and not d.get('exceededTransferLimit'):
            break
    return pd.DataFrame(rows)

if PROP_PATH.exists():
    prop_raw = pd.read_parquet(PROP_PATH)
    print(f"Loaded {len(prop_raw):,} property-database rows from cache")
else:
    prop_raw = _fetch_table(f'{ORG}/Property_Database/FeatureServer/0/query')
    prop_raw.to_parquet(PROP_PATH)
    print(f"Downloaded and cached {len(prop_raw):,} property-database rows")

if PARCEL_PATH.exists():
    parcels = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(parcels):,} parcel polygons from cache")
else:
    feats, off = [], 0
    q = f'{ORG}/FY2026_Parcels/FeatureServer/0/query'
    while True:
        p = dict(where='1=1', outFields='ML,MAP,LOT,LOC_ID,POLY_TYPE', outSR=4326,
                 resultOffset=off, resultRecordCount=2000, f='geojson')
        d = requests.get(q, params=p, timeout=180).json()
        fs = d.get('features', [])
        if not fs:
            break
        feats.extend(fs); off += len(fs)
        if len(fs) < 2000 and not d.get('exceededTransferLimit'):
            break
    parcels = gpd.GeoDataFrame.from_features(feats, crs='EPSG:4326')
    parcels.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(parcels):,} parcel polygons")

print('property-database columns:', list(prop_raw.columns)[:20], '...')
print('parcel CRS:', parcels.crs, ' valid geom:', f"{parcels.geometry.is_valid.mean()*100:.1f}%")


## Section 3 — Classify and Validate

**Column mapping**

| Concept | Column | Notes |
|---|---|---|
| Land value | `LandValue` | market; `$0` on condo units (token-land artifact) |
| Improvement value | `BuildingValue` | market |
| Total assessed value | `AssessedValue` | land + building |
| Use / class code | `StateClassCode` | MA DOR + Cambridge 4-digit variants |
| Residential-exemption flag | `ResidentialExemption` | `'True'` = owner-occupant exemption applies |
| Parcel id (unit) | `MapLot` | e.g. `177-55-3/37` |
| Building group / join key | `GISID` | e.g. `177-55`, joins to parcels `ML` |
| Lot area | `LandArea` | sqft; `0` on condo units, present on CONDO-BLDG master |

**Assessment ratio:** 100% (Massachusetts assesses at full and fair market value).
**Millage source:** City of Cambridge FY26 Tax Rate Letter (classified rates above).

In [ ]:
# De-duplicate: BldgNum sub-records repeat the parcel-total value; keep one row per PID.
prop = prop_raw.sort_values('BldgNum').drop_duplicates('PID', keep='first').reset_index(drop=True)
prop['cls'] = pd.to_numeric(prop['StateClassCode'], errors='coerce')
for c in ['LandValue', 'BuildingValue', 'AssessedValue', 'LandArea']:
    prop[c] = pd.to_numeric(prop[c], errors='coerce').fillna(0.0)
print(f"Parcels after PID de-dup: {len(prop):,} (from {len(prop_raw):,} rows)")

# Classification. NOTE: Cambridge appends 4-digit residential variants (1014, 1021, 1029,
# 1091, 1094, 1095, 1322, ...) that are numerically >900 but RESIDENTIAL; true exempts are
# 900-999 or 9000-9999. Classify by leading digit, not a naive cls>=900 threshold.
def is_exempt_code(c):
    return (900 <= c < 1000) or (c >= 9000)

def is_res_code(c):
    return (c == 13) or (100 <= c < 200) or (1000 <= c < 1400)

def categorize(c):
    if pd.isna(c):
        return 'Other'
    c = int(c)
    if is_exempt_code(c):
        return 'Exempt'
    if c in (101, 1014, 1091, 106): return 'Single Family Residential'
    if c in (102, 1021, 1028, 1029, 1067, 199): return 'Condominium'
    if c in (104, 105, 109, 1094, 1095): return 'Small Multi-Family (2-4 units)'
    if c in (111, 112, 113, 114, 1098): return 'Large Multi-Family (5+ units)'
    if c in (121, 122, 123, 140): return 'Other Residential'
    if c in (130, 131, 132, 1061, 1062, 1322): return 'Vacant Land'
    if c == 13: return 'Mixed Use'
    if is_res_code(c): return 'Other Residential'
    if c == 31: return 'Mixed Use'
    if c in (300, 302, 304): return 'Hotel'
    if c in (340, 341, 342, 343, 345, 346): return 'Office / Commercial Condo'
    if c in (323, 324, 325, 326, 327, 330, 353, 362, 375): return 'Retail / General Commercial'
    if c in (336, 337, 339): return 'Transportation - Parking'
    if c in (390, 391, 392, 440, 442): return 'Vacant Land'
    if 400 <= c < 500 or c == 41: return 'Industrial'
    if 300 <= c < 400: return 'Other Commercial'
    return 'Other'

prop['PROPERTY_CATEGORY'] = prop['cls'].map(categorize)
prop['full_exmp'] = prop['cls'].map(lambda c: is_exempt_code(int(c)) if pd.notna(c) else False).astype(int)
prop['is_res'] = prop['cls'].map(lambda c: is_res_code(int(c)) if pd.notna(c) else False)
prop['is_taxable'] = prop['full_exmp'] == 0

print(prop['PROPERTY_CATEGORY'].value_counts())

# Reconcile the taxable base against the official DOR class totals (assessed-value control).
_res_av = prop.loc[prop.is_res & prop.is_taxable, 'AssessedValue'].sum()
_cip_av = prop.loc[~prop.is_res & prop.is_taxable, 'AssessedValue'].sum()
print(f"\nResidential taxable AV modeled: ${_res_av:,.0f}  (official $41,167,007,215; "
      f"{(_res_av/41_167_007_215-1)*100:+.1f}%)")
print(f"CIP (real) taxable AV modeled:  ${_cip_av:,.0f}  (official comm+ind $29,388,795,880; "
      f"{(_cip_av/29_388_795_880-1)*100:+.1f}%)")


## Section 4 — Current Tax Model (classified, with residential exemption)

Residential parcels pay `$6.67/$1,000` on assessed value net of the FY26 owner-occupant
**residential exemption** ($510,208 off assessed value where the parcel is flagged). CIP
parcels pay `$14.07/$1,000` on assessed value. Fully-exempt parcels (class 900-999 / 9000-9999)
are held out. Validation target is the real-property levy (~$642.9M).

In [ ]:
prop['resexempt'] = (prop['ResidentialExemption'] == 'True') & prop['is_res']
res_taxable_av = np.where(prop['resexempt'],
                          (prop['AssessedValue'] - RES_EXEMPTION).clip(lower=0).values,
                          prop['AssessedValue'].values)

prop['current_tax'] = 0.0
m_res = prop['is_res'] & prop['is_taxable']
m_cip = (~prop['is_res']) & prop['is_taxable']
prop.loc[m_res, 'current_tax'] = res_taxable_av[m_res.values] * RES_RATE
prop.loc[m_cip, 'current_tax'] = prop.loc[m_cip, 'AssessedValue'] * CIP_RATE

res_levy = prop.loc[m_res, 'current_tax'].sum()
cip_levy = prop.loc[m_cip, 'current_tax'].sum()
current_revenue = prop['current_tax'].sum()

print(f"Residential levy modeled: ${res_levy:,.0f}  (official 33.801% x levy = ${0.338010*OFFICIAL_TOTAL_LEVY:,.0f})")
print(f"CIP (real) levy modeled:  ${cip_levy:,.0f}  (official 66.199% x levy - personal-prop tax = ${0.661990*OFFICIAL_TOTAL_LEVY-PERSONAL_PROP_TAX:,.0f})")
gap = (current_revenue / REAL_PROPERTY_TARGET - 1) * 100
print(f"\nTotal real-property modeled: ${current_revenue:,.0f}")
print(f"Target (levy - personal-prop tax): ${REAL_PROPERTY_TARGET:,.0f}")
print(f"Gap: {gap:+.2f}%")
assert abs(gap) < 5.0, f"Revenue gap {gap:.1f}% exceeds threshold"
n_resex = int(prop['resexempt'].sum())
print(f"Parcels receiving residential exemption: {n_resex:,}")


## Section 4b — Condo land imputation

Condo units carry `$0` land / full value in "building" with `LandArea=0`; the land sits on the
class-199 CONDO-BLDG master (lot area present, `$0` value). Impute land **density-aware**:
building land = lot area x citywide residential land `$/sqft`, distributed to units by value share,
capped at 70% of building value so no unit gets a negative building value. Where a unit has no
CONDO-BLDG lot match, fall back to the citywide median land share of non-condo parcels of the
same class (St. Paul value-proportional precedent).

In [ ]:
prop['is_condo_unit'] = (prop['LandValue'] <= 0) & (prop['BuildingValue'] > 0) & prop['is_taxable']

_ncr = prop[prop['is_res'] & ~prop['is_condo_unit'] & (prop['LandValue'] > 0) & (prop['LandArea'] > 0)]
RES_LAND_PSF = float((_ncr['LandValue'] / _ncr['LandArea']).median())
_nc_res = prop[prop['is_res'] & ~prop['is_condo_unit'] & (prop['AssessedValue'] > 0) & (prop['LandValue'] > 0)]
_nc_cip = prop[~prop['is_res'] & prop['is_taxable'] & ~prop['is_condo_unit'] & (prop['AssessedValue'] > 0) & (prop['LandValue'] > 0)]
res_land_share = float((_nc_res['LandValue'] / _nc_res['AssessedValue']).median())
cip_land_share = float((_nc_cip['LandValue'] / _nc_cip['AssessedValue']).median())
print(f"Residential land $/sqft (citywide median): {RES_LAND_PSF:.1f}")
print(f"Fallback land share  residential {res_land_share:.3f}  CIP {cip_land_share:.3f}")
print(f"Condo units to impute: {int(prop['is_condo_unit'].sum()):,}")

_bldg_lot = prop[prop['cls'] == 199].groupby('GISID')['LandArea'].max()
_cond = prop[prop['is_condo_unit']]
_bldg_av = _cond.groupby('GISID')['AssessedValue'].sum()
_lot = _bldg_lot.reindex(_bldg_av.index)
_imp_land_bldg = np.minimum(_lot * RES_LAND_PSF, 0.70 * _bldg_av)

prop['_bldg_av'] = prop['GISID'].map(_bldg_av)
prop['_imp_land_bldg'] = prop['GISID'].map(_imp_land_bldg)
_unit_land_density = prop['AssessedValue'] / prop['_bldg_av'] * prop['_imp_land_bldg']
_fallback_share = np.where(prop['is_res'], res_land_share, cip_land_share)
_unit_land_fallback = prop['AssessedValue'] * _fallback_share
_has_density = prop['_imp_land_bldg'].notna() & (prop['_bldg_av'] > 0)
_condo_land = np.where(_has_density, _unit_land_density, _unit_land_fallback)

prop['taxable_land_value'] = np.clip(np.where(prop['is_condo_unit'], _condo_land, prop['LandValue']), 0, None)
prop['taxable_improvement_value'] = np.clip(
    np.where(prop['is_condo_unit'], prop['AssessedValue'] - _condo_land, prop['BuildingValue']), 0, None)

_dens = int((prop['is_condo_unit'] & _has_density).sum())
print(f"Condo units imputed density-aware: {_dens:,}  fallback: {int(prop['is_condo_unit'].sum())-_dens:,}")


## Section 5 — Split-Rate Model (4:1, classified dual solve)

The reform is revenue-neutral **within each class**, preserving the FY26 residential/CIP split.
The residential exemption is preserved in the reform via `exemption_col` (applied to improvement
first, then land). Fully-exempt parcels are excluded from the solve, the export, and the charts.

In [ ]:
prop['res_exempt_amt'] = np.where(prop['resexempt'], float(RES_EXEMPTION), 0.0)

n_exempt = int((prop['full_exmp'] == 1).sum())
res_df = prop[m_res].copy()
cip_df = prop[m_cip].copy()

r_land, r_imp, r_rev, res_out = model_split_rate_tax(
    df=res_df, land_value_col='taxable_land_value', improvement_value_col='taxable_improvement_value',
    current_revenue=res_df['current_tax'].sum(), land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_col='res_exempt_amt')
c_land, c_imp, c_rev, cip_out = model_split_rate_tax(
    df=cip_df, land_value_col='taxable_land_value', improvement_value_col='taxable_improvement_value',
    current_revenue=cip_df['current_tax'].sum(), land_improvement_ratio=LAND_IMPROVEMENT_RATIO)

# Headline millage reported in the standard export = residential class pair (homeowner-facing).
land_millage, improvement_millage = r_land, r_imp

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Residential split-rate: land {r_land:.4f}  improvement {r_imp:.4f}  (ratio {r_land/r_imp:.2f})")
print(f"CIP split-rate:         land {c_land:.4f}  improvement {c_imp:.4f}  (ratio {c_land/c_imp:.2f})")

gdf = pd.concat([res_out, cip_out]).reset_index(drop=True)
print(f"\nRevenue-neutral check: current ${gdf['current_tax'].sum():,.0f}  new ${gdf['new_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY', current_tax_col='current_tax', new_tax_col='new_tax')
print_category_tax_summary(category_summary, title='cambridge - 4:1 Split-Rate (classified) Tax Impact')

cut = (gdf['tax_change'] < 0).mean() * 100
inc = (gdf['tax_change'] > 0).mean() * 100
within10 = (gdf['tax_change_pct'].abs() <= 10).mean() * 100
print(f"\nParcels with a CUT: {cut:.1f}%   INCREASE: {inc:.1f}%   within +/-10%: {within10:.1f}%")


## Section 5b — Attach geometry

Join the modeled parcels to polygons on `GISID` -> `ML`. Condo units in one building share the
building's polygon (geometry repeats across units).

In [ ]:
_geo = parcels.dropna(subset=['ML']).drop_duplicates('ML')[['ML', 'geometry']]
gdf = gdf.merge(_geo, left_on='GISID', right_on='ML', how='left')
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
matched = gdf.geometry.notna().mean() * 100
print(f"Geometry matched to modeled parcels: {matched:.1f}%")
# Rows with no polygon cannot be mapped or census-joined; keep for the CSV via a separate frame.
gdf_nogeo = gdf[gdf.geometry.isna()].copy()
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Modeled parcels with geometry: {len(gdf):,}  without: {len(gdf_nogeo):,}")


## Section 6 — Exploration chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
summary.plot.barh(ax=ax)
ax.set_title('cambridge - Median Tax Change % by Category (4:1 Split-Rate, classified)')
ax.set_xlabel('Median % Change')
ax.axvline(0, color='k', lw=0.8)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')


## Section 7 — Census Join + Export (canonical closing pattern)

In [ ]:
# Census join - must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out - skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')


In [ ]:
# Export - gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report - PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export - GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")


## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | modeled real-property levy vs. target (official levy - personal-property tax); see Section 4 |
| Classification | residential = class 13 + 1xx + 10xx/13xx variants; CIP = 31/41 + 3xx/4xx; exempt = 9xx/9xxx |
| De-duplication | one row per `PID` (multi-building sub-records repeat the parcel-total value) |
| Condo land | density-aware imputation from CONDO-BLDG lot area x residential land $/sqft |
| Census coverage | see Section 7 |

**Known limitations.** (1) Condo land is imputed at the **citywide** residential land `$/sqft`;
in dense high-land-value cores this understates condo land, so the modeled residential shift
**from condos toward single-family homes is an upper bound**. (2) The parcel extract totals a few
percent below the certified DOR class valuations (benign live-file drift); revenue neutrality is
maintained against the modeled current tax, so the split-rate distribution is unaffected.
(3) Personal property (no land/building) is outside the LVT base by construction.
